# Set Up pwd and auto updates

In [ ]:
from pathlib import Path
import os
import subprocess
# Get the top-level directory of the current git repo
PROJECT_ROOT = Path(
    subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"], text=True
    ).strip()
)

os.chdir(PROJECT_ROOT)


# Enable auto-reloading of custom modules
%load_ext autoreload
%autoreload 2

%pwd


In [ ]:

from season.configs import summarize_config
from collect_external_data.expected_counts import get_expected_counts
from collect_external_data.road_geom import get_road_geometry

from season.persons import SeasonPerson
from season.configs import ScheduleSpecs, SeasonConfig, DayParams, make_season_config, PopulationParams
from traffic.model.bus_system_cost import BusCostConfig

from season.season_orchestrator import SeasonOrchestrator
from traffic.model.hybrid_collector import DataCollectionConfig, Tier1Config, Tier2Config
from traffic.model.tolling import (
    TollConfig, 
    VolumeSignal, 
    FlowSignal, 
    PiecewiseLinearTransform, 
    StepTransform, 
    PITransform
)

import numpy as np
import pandas as pd
from pathlib import Path

from scipy.stats import norm, lognorm, skewnorm, truncnorm, uniform

import seaborn as sns
pd.set_option('display.max_columns', None)


# Ensure Data Exists 

In [ ]:
get_road_geometry()
get_expected_counts()


# Defining the Population Perams 

for small runs: 
- PopulationParams.population_size == make_season_config.max_persons & small n 
- 


In [ ]:
config = make_season_config(
    # ── Identity ──
    season_id='no_toll',
    run_description='',
    seed=33,

    # ── Simulation bounds ──
    n_days=3,
    max_steps=99999,
    start_hr=7,

    # ── Schedules (day-varying) ──
    traffic_percentile_schedule=ScheduleSpecs(mode='static', value=85),
    bus_interval_schedule=ScheduleSpecs(mode='static', value=30),
    crashes_schedule=ScheduleSpecs(mode='static', value=0),
    canyon_closures_schedule=None,

    # ── Population ──
    population_params=PopulationParams(
        population_size=1500,
        prior_car=22.0,
        prior_bus=40.0,
        time_decay_rate=0.1,
        prior_weight=1.0,
        uncertainty_multiplier=1.0,
    ),

    # ── Tolling ──
    toll=TollConfig( _static_toll=0
        # signal=VolumeSignal(),
        # transform=PITransform(target=300, kp=0.5, ki=0.05, toll_min=0, toll_max=50),
        # update_every_n_steps=60,
        # rounding=0.10,
    ),
    bus_user_fee=0.0,
    bus_capacity=60,
    
    # ── Bus Cost ──
    bus_cost_config=BusCostConfig.default(),
    
    # ── Data collection ──
        data_collection=DataCollectionConfig(
        tier1=Tier1Config(
            interval=60,
            scalars=[
                'step', 'p_generate', 'current_toll', 'vehicle_count', 'active_cars', 'bus_riders_waiting',
                'persons_finished', 'persons_pool_remaining', 'persons_in_transit',
            ],
            window_scalars=[
                'recent_travel_time_avg',
                'rolling_count_vehicles_generated',
                'rolling_count_persons_generated',
            ],
            histograms=['implicit_sl_delta'],
            window_seconds=600,
        ),
        # tier2, tier3, tier4 default to None/False (off)
    ),


)


summarize_config(config, high_only=False)

In [ ]:

cfg = make_season_config(
    season_id="post_data_collect_test",
    run_description="",
    seed=42,
    n_days=3,
    max_concurrent_vehicles=5000,
    max_steps=30000,
    max_persons=99999,
    start_hr=7,
    bus_capacity=60,
    traffic_percentile_schedule=ScheduleSpecs("static", 85),
    bus_interval_schedule=ScheduleSpecs("static", 30),
    crashes_schedule=ScheduleSpecs("static", 0),
    toll=TollConfig.static(car=0.0),
    bus_user_fee=0.0,
    population_params=PopulationParams(
        population_size=1500,
        prior_bus=60.0,
    ),
    data_collection=DataCollectionConfig(
    tier1=Tier1Config(
        interval=60,
        scalars=[
            "step", "current_toll", "vehicle_count", "active_cars", "active_buses",
            "persons_at_bus_stop", "persons_finished", "persons_pool_remaining", "persons_in_transit",
        ],
        window_scalars=[
            "bus_mode_share_recent",
            "recent_travel_time_avg",
            "rolling_count_vehicles_generated",
            "rolling_count_persons_generated",
        ],
        histograms=[],
        window_seconds=300,
    ),
    tier2=Tier2Config(
        sample_interval=2,
        max_samples=1000,
        max_agents_per_sample=1000,
    ),
    # tier3 and tier4 off by default
),

)

summarize_config(cfg, high_only=False)



In [ ]:
orchestrator = SeasonOrchestrator(season_config=cfg)
orchestrator.run_season()



In [ ]:
# Read the day_0_model_ts parquet produced by the season run
parquet_path = PROJECT_ROOT / "data" / "season_outputs" / "speed_test2" / "day_2_model_ts.parquet"

if not parquet_path.exists():
    raise FileNotFoundError(f"Parquet file not found: {parquet_path}")

df_day = pd.read_parquet(parquet_path)

print(f"Loaded: {parquet_path}")
print("Shape:", df_day.shape)
print("\nColumn dtypes:")
print(df_day.dtypes)

# Show a quick sample
try:
    display(df_day.head(10))
except NameError:
    print(df_day.head(10))

In [ ]:
df_day

In [ ]:
orchestrator.last_model_run

In [ ]:
import cProfile
import pstats

# some stuff used for optimization

def main():
    # Example usage of SeasonOrchestrator with example_config
    orchestrator = SeasonOrchestrator(season_config=config, store_data=True)
    orchestrator.run_season()

if __name__ == "__main__":
    prof = cProfile.Profile()
    prof.enable()

    main()

    prof.disable()
    prof.dump_stats("prof.stats")

    p = pstats.Stats("prof.stats")
    p.strip_dirs().sort_stats("cumulative").print_stats(40)




# Example configs

In [ ]:
# ===================== Example Toll Configurations =====================
# Uncomment and use any of these in make_season_config(toll=..., bus_user_fee=...)

# 1. Static toll (fixed price)
# toll=TollConfig.static(car=10.0),

# 2. Volume-based piecewise linear (current config above)
# toll=TollConfig(
#     signal=VolumeSignal(),
#     transform=PiecewiseLinearTransform(threshold=100, slope=0.05, base=5.0),
#     update_every_n_steps=60,
#     rounding=0.25,
# ),

# 3. Flow-based piecewise linear (rolling average arrival rate)
# toll=TollConfig(
#     signal=FlowSignal(window_steps=300),  # 5-minute rolling window
#     transform=PiecewiseLinearTransform(threshold=1.0, slope=10.0, base=2.0),
#     update_every_n_steps=60,
#     rounding=0.25,
#     cap=25.0,  # max toll $25
# ),

# 4. Volume-based step toll (binary: $0 or $10)
# toll=TollConfig(
#     signal=VolumeSignal(),
#     transform=StepTransform(threshold=100, toll=10.0),
#     update_every_n_steps=60,
# ),

# 5. Volume-based PI controller (feedback-driven)
# toll=TollConfig(
#     signal=VolumeSignal(),
#     transform=PITransform(target=300, kp=0.5, ki=0.05, toll_min=0, toll_max=50),
#     update_every_n_steps=60,
#     rounding=0.10,
# ),

## Testing Tolling

In [ ]:
%matplotlib inline
from traffic.model.tolling import (
    TollConfig, VolumeSignal,FlowSignal, PITransform,
    PiecewiseLinearTransform, StepTransform,
    interactive_toll_plot,
)

# Compare PI controllers
interactive_toll_plot(
    toll_configs=[
        TollConfig(signal=FlowSignal(), transform=PITransform(target=.2, kp=10, ki=20, toll_min=0, toll_max=100), update_every_n_steps=60),
        TollConfig(signal=FlowSignal(), transform=PITransform(target=.25, kp=10, ki=20, toll_min=0, toll_max=100), update_every_n_steps=60),
        # TollConfig(signal=VolumeSignal(), transform=PITransform(target=200, kp=0.8, ki=0.08, toll_min=0, toll_max=50), update_every_n_steps=60),
    ],
    signal_range=(0.0, 1.0),
    signal_label="Flow (veh/step)",
    labels=["PI loose (t=200)", "PI loose (t=150)"],
)


In [ ]:
TollConfig(signal=FlowSignal(), transform=PITransform(target=.2, kp=10, ki=20, toll_min=0, toll_max=100), update_every_n_steps=60),
        TollConfig(signal=FlowSignal(), transform=PITransform(target=.25, kp=10, ki=20, toll_min=0, toll_max=100), update_every_n_steps=60),